# **1. Техническое задание (ТЗ)**  

### **1.1. Входные данные**
В качестве входных данных используется датасет изображений, состоящий из двух классов:  
- **cats/** — фотографии кошек  
- **dogs/** — фотографии собак  

Каждое изображение:  
- имеет произвольное разрешение;  
- может быть сделано в разных условиях (фон, освещение, ракурс);  
- содержит ровно одного объекта (кошку или собаку).  

Датасет представляет собой классическую задачу **бинарной классификации изображений**.

### **1.2. Цель задачи**
Построить модель компьютерного зрения, которая по входному изображению должна определить, к какому классу оно относится:
- **0 — Cat**
- **1 — Dog**

Выход модели:  
- либо вероятность принадлежности к классу "Dog" (0–1),  
- либо бинарное решение после порога: `dog если p > 0.5`.

### **1.3. Особенности предметной области**
- Изображения **не стандартизированы**, что требует предварительной обработки:
  - приведение к одному размеру,
  - нормализация
- Данные могут быть **несбалансированы**, необходимо это проверить.

# **2. Обзор подходов к решению задач классификации изображений**

Существуют два основных подхода:

### **Подход 1 — Обучение модели “с нуля”**
Создаётся собственная архитектура нейронной сети, например:
- 2–4 сверточных слоя,
- пулинг,
- полносвязные слои

Преимущества:  
- максимально понятен для учебных целей,  
- полностью контролируемая архитектура.

Недостатки:  
- требует большого датасета для хороших результатов,   
- качество обычно хуже, чем у моделей-гигантов.

Используемые инструменты:
- **PyTorch** (`torch.nn`, `torchvision.datasets`, `torchvision.transforms`)
- **NumPy**, **OpenCV** для подготовки данных

### **Подход 2 — Transfer Learning (предобученные модели)**

Суть:
- взять модель, предобученную на **ImageNet**,
- заменить финальный слой под задачу "cats vs dogs",
- дообучить (fine-tuning) на своей выборке.

Самые популярные модели:
- **ResNet-18/34/50**  
- **MobileNetV2 / V3**  
- **EfficientNet-B0/B1**  
- **VGG-16 / VGG-19**  
- **DenseNet121**  

Преимущества:
- высокая точность даже на малом датасете,
- намного лучше устойчивость к “шумным” данным.

Недостатки:
- модель тяжелее,
- меньше прозрачности архитектуры.

Инструменты:
- PyTorch: `torchvision.models`
- TensorFlow: `tensorflow.keras.applications`
- ONNX, OpenVINO для оптимизации

In [104]:
import os
import random
import numpy as np
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

IMG_SIZE = 128             
BATCH_SIZE = 32
TEST_SIZE = 0.2
RANDOM_SEED = 42           
LR = 0.001
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

full_dataset = datasets.ImageFolder(root="data", transform=transform)
print("Классы:", full_dataset.classes)            
print("Всего изображений:", len(full_dataset))

targets = [s[1] for s in full_dataset.samples]  
print("Распределение:", Counter(targets))

Классы: ['cats', 'dogs']
Всего изображений: 24998
Распределение: Counter({0: 12499, 1: 12499})


In [105]:
indices = list(range(len(full_dataset)))
y = [s[1] for s in full_dataset.samples]

train_idx, test_idx = train_test_split(indices, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)

train_dataset = Subset(full_dataset, train_idx)
test_dataset  = Subset(full_dataset, test_idx)

print("Train size:", len(train_dataset))
print("Test size: ", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)


Train size: 19998
Test size:  5000


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),  
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             

            nn.Conv2d(16, 32, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             

            nn.Conv2d(32, 64, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                            
        )
     
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.classifier = nn.Sequential(
            nn.Flatten(),        
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)  
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x  


In [ ]:
model = SimpleCNN(num_classes=1).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): ReLU()
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [ ]:
from sklearn.metrics import accuracy_score

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).float().view(-1, 1)  

        optimizer.zero_grad()
        logits = model(images)                
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()  
        preds = (probs >= 0.5).astype(int)
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.cpu().numpy().ravel().astype(int).tolist())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc  = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).float().view(-1, 1)
            logits = model(images)
            loss = criterion(logits, labels)
            running_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            preds = (probs >= 0.5).astype(int)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().ravel().astype(int).tolist())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc  = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


In [ ]:
images, labels = next(iter(train_loader))
print("batch images shape:", images.shape)
print("batch labels shape:", labels.shape)

with torch.no_grad():
    out = model(images.to(DEVICE))
print("output shape:", out.shape)  


batch images shape: torch.Size([32, 3, 128, 128])
batch labels shape: torch.Size([32])
output shape: torch.Size([32, 1])


In [ ]:
epoch = 10
train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
val_loss, val_acc = evaluate(model, test_loader, criterion, DEVICE)

torch.save(model.state_dict(), "best_simplecnn.pth")

print(f"Epoch [{epoch}] "
        f"Train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f}, acc: {val_acc:.4f} ")


Epoch [10/10] Train loss: 0.5820, acc: 0.6925 | Val loss: 0.5946, acc: 0.6638 


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import accuracy_score

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)  

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features         
model.fc = nn.Linear(num_features, 2)        

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)  

def train_one_epoch_resnet(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    preds = []
    labels_list = []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)                 
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds.extend(outputs.argmax(dim=1).cpu().numpy())
        labels_list.extend(labels.cpu().numpy())

    return total_loss / len(loader.dataset), accuracy_score(labels_list, preds)

def evaluate_resnet(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    preds = []
    labels_list = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(labels_list, preds)

epoch = 3
train_loss, train_acc = train_one_epoch_resnet(model, train_loader, criterion, optimizer, DEVICE)
val_loss, val_acc     = evaluate_resnet(model, test_loader, criterion, DEVICE)
print(f"Epoch {epoch} | train_acc={train_acc:.4f}, val_acc={val_acc:.4f} | train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
torch.save(model.state_dict(), "best_resnet18_finetuned_head.pth")



c:\Users\CifronPro\AppData\Local\Programs\Python\Python313\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 3 | train_acc=0.8914, val_acc=0.9294 | train_loss=0.2468, val_loss=0.1703
